# Gathering preliminary info about the Glassdoor reviews dataset

In [ ]:
# Import libraries
import pandas as pd

# Load the dataset
data = pd.read_csv("glassdoor_data_15_reviews_manual.csv")

# Preview the data
print(data.head())

# Get info about the columns
print(data.info())

# Check number of rows and columns
print(data.shape)

# Get statistical summary of numeric columns
print(data.describe())

   review_id               rating_date  count_helpful  count_unhelpful  \
0          1  2025-06-03T00:00:00.000Z              0                0   
1          2  2025-05-29T00:00:00.000Z              0                0   
2          3  2025-05-28T00:00:00.000Z              0                0   
3          4  2025-05-26T00:00:00.000Z              0                0   
4          5  2025-05-23T00:00:00.000Z              0                0   

   employee_length employee_status                       employee_type  \
0              0.0         REGULAR                    Current employee   
1              1.0        CONTRACT  Current employee, less than 1 year   
2              1.0         REGULAR  Current employee, less than 1 year   
3              2.0       PART_TIME                   Current employees   
4              0.0         REGULAR                    Current employee   

   flag_covid  flag_featured flags_business_outlook  ... review_advice  \
0       False          False        

# Standardize job titles

In [ ]:
# Inspect the values of existing column
print(data["employee_job_title"].value_counts())

employee_job_title
Warehouse Associate      19
Associate                18
Warehouse Worker          6
Fulfillment Associate     5
FC Associate I            5
                         ..
Area Manager II           1
Learning Coordinator      1
Anonymous Employee        1
Out Bound Ship Dock       1
IT Manager II             1
Name: count, Length: 66, dtype: int64


In [ ]:
# Normalize the column by lowercasing + removing extra spaces
s = (data["employee_job_title"]
     .fillna("")
     .astype(str)
     .str.lower()
     .str.strip())

data["employee_job_title"]

,employee_job_title
0,Warehouse Fulfillment Associate
1,Warehouse Associate
2,Warehouse Associate
3,Employee
4,Fulfillment Associate
...,...
135,Warehouse Associate
136,Out Bound Ship Dock
137,Warehouse Associate
138,IT Manager II


In [ ]:
# Bucketing job titles into various buckets

import re

def bucket_title(title):

    # Learning Ambassadaor
    if "learning ambassador" in title:
        return "Learning Ambassador"

    # Stower / Inbound
    elif re.search(r"\bstow|stower", title):
        return "Stower"

    # Picker / Packer / Ship Dock
    elif re.search(r"\bpicker|\bpacker|pick|pack|ship dock|outbound", title):
        return "Picker/Packer"

    # General Associate roles
    elif re.search(r"associate|warehouse worker|team member|fc associate", title):
        return "Associate"

    else:
        return "Management, IT, RME, HR"

In [ ]:
# Apply the bucket function and print how many categories in that column
data["job_title_bucket"] = s.apply(bucket_title)
data["job_title_bucket"].value_counts()

,count
job_title_bucket,
Associate,77
"Management, IT, RME, HR",45
Picker/Packer,13
Stower,5


In [ ]:
# Print the title and job title bucket together
data[["employee_job_title", "job_title_bucket"]]

,employee_job_title,job_title_bucket
0,Warehouse Fulfillment Associate,Associate
1,Warehouse Associate,Associate
2,Warehouse Associate,Associate
3,Employee,"Management, IT, RME, HR"
4,Fulfillment Associate,Associate
...,...,...
135,Warehouse Associate,Associate
136,Out Bound Ship Dock,Picker/Packer
137,Warehouse Associate,Associate
138,IT Manager II,"Management, IT, RME, HR"


# Fixing symbols and typos from summary and text-based review columns

In [ ]:
# Fix symbols and encoding artifacts from
# summary, review pros / cons, and advice to management columns

import html
import re

text_columns = [
    "summary",
    "review_pros",
    "review_cons",
    "advice_to_management"
]

for col in text_columns:
    if col in data.columns:
        data[col] = (
            data[col]
            .astype(str)
            .apply(html.unescape)  # Fix &amp;
            .str.replace("", "'", regex=False)  # Fix bad apostrophe
            .str.replace("â€™", "'", regex=False)  # Another common artifact
            .str.replace("â€œ", '"', regex=False)
            .str.replace("â€", '"', regex=False)
        )

In [ ]:
# Remove newlines, extra spaces, and weird formatting
for col in text_columns:
    if col in data.columns:
        data[col] = (
            data[col]
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )

In [ ]:
# Cleaning up obvious typos
common_typos = {
    "enviorment": "environment",
    "mangement": "management",
    "recieve": "receive",
    "managment": "management",
    "adivce": "advice"
}

for col in text_columns:
    if col in data.columns:
        for wrong, correct in common_typos.items():
            data[col] = data[col].str.replace(wrong, correct, regex=False)

In [ ]:
# Cleaning employee job title:
# Sometimes, titles contain stray spaces or casing issues.
data["employee_job_title"] = (
    data["employee_job_title"]
    .astype(str)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
)

In [ ]:
# Detect remaining strange characters that might have initially missed
for col in text_columns:
    if col in data.columns:
        strange = data[data[col].str.contains(r"[^\x00-\x7F]", regex=True, na=False)]
        print(f"{col} strange characters count:", len(strange))

summary strange characters count: 8
review_pros strange characters count: 6
review_cons strange characters count: 13
advice_to_management strange characters count: 7


In [ ]:
# Replacing weird punctutation marks and etc.
import unicodedata
import re

text_columns = ["summary", "review_pros", "review_cons", "advice_to_management"]

PUNCT_REPLACEMENTS = {
    "’": "'",
    "‘": "'",
    "“": '"',
    "”": '"',
    "–": "-",
    "—": "-"
}

def clean_text(s: str) -> str:
    if s is None:
        return ""
    s = str(s)

    # Replace smart punctuation
    for bad, good in PUNCT_REPLACEMENTS.items():
        s = s.replace(bad, good)

    # Remove emojis (basic unicode ranges)
    s = re.sub(r"[\U00010000-\U0010FFFF]", "", s)

    # Convert accented characters to ASCII (ü->u, é->e, ñ->n, ß->ss, etc.)
    s = unicodedata.normalize("NFKD", s).encode("ascii", "ignore").decode("ascii")

    # Normalize whitespace
    s = re.sub(r"\s+", " ", s).strip()

    return s

for col in text_columns:
    if col in data.columns:
        data[col] = data[col].apply(clean_text)

In [ ]:
# Re-run how many strange characters I get now.
for col in text_columns:
    strange = data[data[col].str.contains(r"[^\x00-\x7F]", regex=True, na=False)]
    print(f"{col} strange characters count:", len(strange))

summary strange characters count: 0
review_pros strange characters count: 0
review_cons strange characters count: 0
advice_to_management strange characters count: 0


# Handling missing values and dealing with column relevancy to analysis

In [ ]:
for col in data.columns:
    print(col, "→ unique values:", data[col].nunique())

review_id → unique values: 140
rating_date → unique values: 102
count_helpful → unique values: 1
count_unhelpful → unique values: 1
employee_length → unique values: 10
employee_status → unique values: 5
employee_type → unique values: 7
flag_covid → unique values: 1
flag_featured → unique values: 1
flags_business_outlook → unique values: 3
flags_ceo_approval → unique values: 4
flags_recommend_friend → unique values: 3
rating_culture_values → unique values: 6
rating_diversity_inclusion → unique values: 6
rating_overall → unique values: 5
rating_work_life → unique values: 6
summary → unique values: 131
company_name → unique values: 1
review_advice → unique values: 0
career_opportunities_rating → unique values: 0
employee_location → unique values: 98
employee_job_title → unique values: 67
advice_to_management → unique values: 46
review_pros → unique values: 139
review_cons → unique values: 139
rating_compensation_benefits → unique values: 6
rating_senior_leadership → unique values: 6
ratin

In [ ]:
# Drop these columns from the dataframe
cols_to_drop = [
    "count_helpful",
    "count_unhelpful",
    "flag_covid",
    "flag_featured",
    "company_name",
    "review_advice",
    "career_opportunities_rating"
]

data = data.drop(columns=cols_to_drop)

In [ ]:
# Audit missing values
data.isnull().sum().sort_values(ascending=False)

,0
flags_business_outlook,72
flags_ceo_approval,71
flags_recommend_friend,57
employee_location,8
employee_status,2
employee_length,1
employee_type,0
rating_date,0
review_id,0
rating_culture_values,0


# Replacing missing values for employee status, location, and length to unknown

In [ ]:
# Fill in missing values for employee status & location to Unknown
# Fill in missing employee length to its median

data["employee_length"] = data["employee_length"].fillna(data["employee_length"].median())
data["employee_status"] = data["employee_status"].fillna("Unknown")
data["employee_location"] = data["employee_location"].fillna("Unknown")

# Resolving datetime issues

In [ ]:
# Fix rating date column by converting to datetime format
data["rating_date"] = pd.to_datetime(
    data["rating_date"],
    errors="coerce",
    utc=True
)

# Remove timezone + time portion but keep datetime type
data["rating_date"] = data["rating_date"].dt.tz_convert(None).dt.normalize()

# Now create derived columns
data["review_year"] = data["rating_date"].dt.year
data["review_month"] = data["rating_date"].dt.to_period("M")

# Resolving employee-related columns (text standardization and missing values)

In [ ]:
# Standardize employee category columns
cat_cols = ["employee_status", "employee_type"]

for col in cat_cols:
    if col in data.columns:
        data[col] = (
            data[col]
            .astype(str)
            .str.strip()
            .str.replace("_", " ", regex=False)
            .str.replace(r"\s+", " ", regex=True)
            .str.title()
        )

# Convert to categorical type
for col in cat_cols:
    if col in data.columns:
        data[col] = data[col].astype("category")

# Replace employee location, job title, and advice missing values to unknown
# Drop the job title column completely
data["employee_job_title"] = data["employee_job_title"].fillna("Unknown")

if "employee_job_title" in data.columns:
    data = data.drop(columns=["employee_job_title"])

data["employee_location"] = data["employee_location"].fillna("Unknown")
data["advice_to_management"] = data["advice_to_management"].fillna("No Response")

In [ ]:
# Clean text columns consistently (to prevent broken NLP later)
import html
import re

text_cols = ["summary", "review_pros", "review_cons", "advice_to_management"]

for col in text_cols:
    if col in data.columns:
        data[col] = (
            data[col].fillna("")
            .astype(str)
            .apply(html.unescape)
            .str.replace(r"\s+", " ", regex=True)
            .str.strip()
        )
        data.loc[data[col].str.lower().isin(["nan", "none", "null", ""]), col] = ""

# Text standardization for flag columns

In [ ]:
# Standardize casings for flag business, CEO, and recommend columns
flag_cols = [
    "flags_business_outlook",
    "flags_ceo_approval",
    "flags_recommend_friend"
]

for col in flag_cols:
    if col in data.columns:
        # Replace true NaN first
        data[col] = data[col].fillna("No Response")

        # Then clean formatting
        data[col] = (
            data[col]
            .astype(str)
            .str.strip()
            .str.replace("_", " ", regex=False)
            .str.title()
        )

        # Just in case any string "nan" slipped in earlier
        data[col] = data[col].replace("Nan", "No Response")

# Print final cleaned dataframe

In [ ]:
# Define the new order
new_order = [
    # Identifiers
    "review_id",

    # Time
    "rating_date",
    "review_year",
    "review_month",

    # Employee Metadata
    "employee_status",
    "employee_type",
    "employee_length",
    "employee_location",
    "job_title_bucket",

    # Flags
    "flags_business_outlook",
    "flags_ceo_approval",
    "flags_recommend_friend",

    # Ratings
    "rating_overall",
    "rating_work_life",
    "rating_compensation_benefits",
    "rating_senior_leadership",
    "rating_career_opportunities",
    "rating_culture_values",
    "rating_diversity_inclusion",

    # Text
    "summary",
    "review_pros",
    "review_cons",
    "advice_to_management"
]

# Keep only columns that actually exist
new_order = [col for col in new_order if col in data.columns]
data = data[new_order]

In [ ]:
# Print final cleaned dataframe in the desired order
print(data.head(10))

   review_id rating_date  review_year review_month employee_status  \
0          1  2025-06-03         2025      2025-06         Regular   
1          2  2025-05-29         2025      2025-05        Contract   
2          3  2025-05-28         2025      2025-05         Regular   
3          4  2025-05-26         2025      2025-05       Part Time   
4          5  2025-05-23         2025      2025-05         Regular   
5          6  2025-05-18         2025      2025-05         Regular   
6          7  2025-05-17         2025      2025-05         Regular   
7          8  2025-05-15         2025      2025-05         Regular   
8          9  2025-05-14         2025      2025-05         Regular   
9         10  2025-05-14         2025      2025-05         Unknown   

                        employee_type  employee_length  employee_location  \
0                    Current Employee              0.0       Richmond, VA   
1  Current Employee, Less Than 1 Year              1.0       Brampton, ON  

In [ ]:
# Print finalized dataframe into a CSV file
data.to_csv("glassdoor_data_15_reviews_cleaned.csv", index=False)